In [1]:
import fastf1
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore') 

print("🏎️ Initializing Real FastF1 Data Extraction (Brazil 2023)...")

fastf1.Cache.enable_cache('f1_cache')
session = fastf1.get_session(2023, 'Brazil', 'R')
session.load(telemetry=False, weather=False)

laps = session.laps.copy()

# 1. FILTERING: Remove Lap 1 (Chaos) and Pit Laps
valid_laps = laps[(laps['LapNumber'] > 1) & (laps['PitOutTime'].isnull()) & (laps['PitInTime'].isnull())]

# 2. OVERTAKE DETECTION: If the track position has changed then it means they have overtaken someone
valid_laps['Pos_Change'] = valid_laps.groupby('Driver')['Position'].diff()
valid_laps['Overtake_Success'] = (valid_laps['Pos_Change'] < 0).astype(int)

# 3. SPEED ADVANTAGE: Speed Trap compared to the field average
lap_avg_speed = valid_laps.groupby('LapNumber')['SpeedST'].transform('mean')
valid_laps['Speed_Advantage_kmh'] = valid_laps['SpeedST'] - lap_avg_speed

# 4. FINAL CLEANUP
df_real = valid_laps[['Speed_Advantage_kmh', 'TyreLife', 'Overtake_Success']].dropna()
df_real.rename(columns={'TyreLife': 'Tire_Age'}, inplace=True)

print("\n✅ Real Data Extracted! Distribution of Actual Overtakes:")
print(df_real['Overtake_Success'].value_counts())
print("-" * 50)

X = df_real.drop('Overtake_Success', axis=1)
y = df_real['Overtake_Success']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Junior Model (No SMOTE)
rf_junior = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_junior.fit(X_train, y_train)
print("\n🚨 JUNIOR EVALUATION (Without SMOTE):")
print(classification_report(y_test, rf_junior.predict(X_test)))

# Senior Model (With SMOTE)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

rf_senior = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_senior.fit(X_train_smote, y_train_smote)
print("\n🏆 SENIOR EVALUATION (With SMOTE):")
print(classification_report(y_test, rf_senior.predict(X_test)))

🏎️ Initializing Real FastF1 Data Extraction (Brazil 2023)...


core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No 


✅ Real Data Extracted! Distribution of Actual Overtakes:
Overtake_Success
0    879
1    107
Name: count, dtype: int64
--------------------------------------------------

🚨 JUNIOR EVALUATION (Without SMOTE):
              precision    recall  f1-score   support

           0       0.90      0.98      0.94       177
           1       0.25      0.05      0.08        21

    accuracy                           0.88       198
   macro avg       0.57      0.52      0.51       198
weighted avg       0.83      0.88      0.85       198


🏆 SENIOR EVALUATION (With SMOTE):
              precision    recall  f1-score   support

           0       0.91      0.82      0.86       177
           1       0.16      0.29      0.21        21

    accuracy                           0.77       198
   macro avg       0.53      0.56      0.54       198
weighted avg       0.83      0.77      0.79       198



### The model does improve due to smote but, still misses out many valid overtakes

Junior Recall: 0.05. Out of 21 actual overtakes in the test set, the AI only caught 1.

Senior Recall: 0.29. Out of 21 actual overtakes, the AI caught 6.

This is due to lack features, we need Advanced Feature Engineering of features like:

Is_In_DRS_Train (Boolean: 1 or 0)

Distance_To_Car_Ahead_At_Corner_Exit (Meters)

Defending_Car_Tire_Age (Laps)

but now, I can only engineer one feature called Pace_Advantage_Sec as of now.

In [ ]:

valid_laps = laps[(laps['LapNumber'] > 1) & (laps['PitOutTime'].isnull()) & (laps['PitInTime'].isnull())]
valid_laps['Pos_Change'] = valid_laps.groupby('Driver')['Position'].diff()
valid_laps['Overtake_Success'] = (valid_laps['Pos_Change'] < 0).astype(int)

lap_avg_speed = valid_laps.groupby('LapNumber')['SpeedST'].transform('mean')
valid_laps['Speed_Advantage_kmh'] = valid_laps['SpeedST'] - lap_avg_speed

valid_laps['LapTime_Sec'] = valid_laps['LapTime'].dt.total_seconds()
lap_avg_time = valid_laps.groupby('LapNumber')['LapTime_Sec'].transform('mean')
valid_laps['Pace_Advantage_Sec'] = lap_avg_time - valid_laps['LapTime_Sec']

df_real = valid_laps[['Speed_Advantage_kmh', 'TyreLife', 'Pace_Advantage_Sec', 'Overtake_Success']].dropna()
df_real.rename(columns={'TyreLife': 'Tire_Age'}, inplace=True)

X = df_real.drop('Overtake_Success', axis=1)
y = df_real['Overtake_Success']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_junior = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_junior.fit(X_train, y_train)
print(classification_report(y_test, rf_junior.predict(X_test)))

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

rf_senior = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_senior.fit(X_train_smote, y_train_smote)
print(classification_report(y_test, rf_senior.predict(X_test)))

              precision    recall  f1-score   support

           0       0.90      0.98      0.94       177
           1       0.25      0.05      0.08        21

    accuracy                           0.88       198
   macro avg       0.57      0.52      0.51       198
weighted avg       0.83      0.88      0.85       198

              precision    recall  f1-score   support

           0       0.91      0.76      0.82       177
           1       0.14      0.33      0.20        21

    accuracy                           0.71       198
   macro avg       0.52      0.55      0.51       198
weighted avg       0.82      0.71      0.76       198



Recall did improve to 33% after adding one more column.